# 2.2 Runtime context

**Context** is data you pass **into** `invoke` (`context=...`). It is not written by tools the way state is. Typical uses: user id, locale, feature flags — things the host app already knows.

This notebook:

1. Declares a dataclass schema and passes it on invoke (the model still cannot see it unless you expose it).
2. Adds tools that read `runtime.context` so the model can answer from that config.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Context schema

A dataclass (or Pydantic model) with defaults. `create_agent(..., context_schema=ColourContext)` types the `context=` argument.

In [ ]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

Without tools that read context, the model **guesses**. Passing `ColourContext()` is not enough by itself.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [ ]:
from pprint import pprint

# Likely a generic "I don't know" — context is not in the prompt yet
pprint(response)

## Accessing context from tools

`ToolRuntime[ColourContext]` types `runtime.context`. The model calls these tools; your code returns the dataclass fields.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [ ]:
# Defaults: favourite_colour="blue"
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

Override fields per request — same agent, different user preferences, no state write.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)